# 00 - Environment setup (RunPod)

Run this once per pod. It installs dependencies, checks the GPU, runs the test
suite, and pre-downloads the pretrained models so later notebooks never stall
mid-run.

Expected total: about 5 minutes plus roughly 4 GB of downloads.

In [ ]:
!nvidia-smi

In [ ]:
import os, sys
REPO = os.path.abspath(os.path.join(os.getcwd(), "..")) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
os.chdir(REPO)
sys.path.insert(0, os.path.join(REPO, "src"))
os.environ["PYTHONIOENCODING"] = "utf-8"
print("repo:", REPO)

## Install dependencies

PyTorch is normally preinstalled on RunPod images, so this only fills gaps.

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
import torch, transformers, numpy

print("torch       ", torch.__version__)
print("transformers", transformers.__version__)
print("numpy       ", numpy.__version__)
print("cuda        ", torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print("gpu         ", p.name)
    print(f"memory       {p.total_memory / 1024 ** 3:.1f} GB")
    print("bf16         ", torch.cuda.is_bf16_supported())

## Run the test suite

These check causality, KV-cache equivalence, collation alignment and, most
importantly, that the model actually learns homograph disambiguation on a
controlled corpus. All must pass before you spend GPU time.

In [ ]:
import subprocess, sys, os

tests = [
    "tests/test_models.py",
    "tests/test_data.py",
    "tests/test_end_to_end.py",
    "tests/test_integration.py",
]
env = dict(os.environ, PYTHONIOENCODING="utf-8")
for t in tests:
    print()
    print("=" * 60)
    print(t)
    print("=" * 60)
    r = subprocess.run([sys.executable, t], capture_output=True, text=True, env=env)
    print(r.stdout[-2500:])
    if r.returncode != 0:
        print("STDERR:", r.stderr[-2000:])
        raise SystemExit(t + " FAILED - fix this before continuing")
print()
print("All tests passed. The code is ready to train.")

## Pre-download the pretrained models

In [ ]:
import yaml
from transformers import (
    AutoFeatureExtractor, AutoModel, AutoModelForCTC, AutoProcessor, AutoTokenizer, MimiModel,
)

cfg = yaml.safe_load(open("configs/base.yaml", encoding="utf-8"))

print("1/4 CTC aligner")
AutoProcessor.from_pretrained(cfg["align"]["model_id"])
AutoModelForCTC.from_pretrained(cfg["align"]["model_id"])

print("2/4 SSL span encoder")
AutoFeatureExtractor.from_pretrained(cfg["spanemb"]["model_id"])
AutoModel.from_pretrained(cfg["spanemb"]["model_id"])

print("3/4 MARBERTv2 teacher")
AutoTokenizer.from_pretrained(cfg["teacher"]["model_id"])
AutoModel.from_pretrained(cfg["teacher"]["model_id"])

print("4/4 Mimi codec")
MimiModel.from_pretrained(cfg["codec_model_id"])

print()
print("all models cached")

Setup is done. Continue to **01_prepare_data.ipynb**.